# VietHandOCR Part 1: Data Preparation & EDA

Welcome to **Part 1** of the VietHandOCR pipeline. 
- **Next Notebook**: [Part 2: Digital Image Processing (DIP) Pipeline](./02_Digital_Image_Processing.ipynb)

## Introduction
This notebook handles extracting the UIT-HWDB dataset, optimizing memory (downcasting), and performing a strict writer-independent split. We save the intermediate results to `.txt` files for the next notebook.


In [1]:
import os, gc, zipfile, random
import numpy as np
import pandas as pd

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)
print("Seeded everything successfully!")


Seeded everything successfully!


In [2]:
def reduce_mem_usage(df):
    '''Iterates through columns and modifies data types to reduce memory usage.'''
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and not pd.api.types.is_categorical_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max: df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                else: df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max: df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
                else: df[col] = df[col].astype(np.float64)
        else:
            num_unique_values = len(df[col].unique())
            num_total_values = len(df[col])
            if num_unique_values / num_total_values < 0.5:
                df[col] = df[col].astype('category')
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory decreased from {start_mem:.2f}MB to {end_mem:.2f}MB')
    return df


In [3]:
def writer_independent_split(metadata_df, val_size=0.1):
    '''Splits data ensuring writers in validation set are not in training set.'''
    if 'writer_id' not in metadata_df.columns:
        from sklearn.model_selection import train_test_split
        return train_test_split(metadata_df, test_size=val_size, random_state=42)
        
    writers = list(metadata_df['writer_id'].unique())
    np.random.shuffle(writers)
    split_idx = int(len(writers) * (1 - val_size))
    train_writers, val_writers = writers[:split_idx], writers[split_idx:]
    
    train_df = metadata_df[metadata_df['writer_id'].isin(train_writers)].copy()
    val_df = metadata_df[metadata_df['writer_id'].isin(val_writers)].copy()
    return train_df, val_df

import json
import glob

def build_metadata(base_dir):
    '''Crawls the UIT-HWDB structure to build a pandas DataFrame from label.json files.'''
    records = []
    # Search for all label.json files in the dataset
    json_paths = glob.glob(os.path.join(base_dir, '**', 'label.json'), recursive=True)
    
    for json_path in json_paths:
        # Path structure: base_dir / <level> / <split> / <writer_id> / label.json
        parts = json_path.split(os.sep)
        if len(parts) >= 4:
            level = parts[-4]
            split = parts[-3]
            writer_id = parts[-2]
            
            with open(json_path, 'r', encoding='utf-8') as f:
                try:
                    labels = json.load(f)
                    for img_name, text in labels.items():
                        img_path = os.path.join(os.path.dirname(json_path), img_name)
                        records.append({
                            'image_path': img_path,
                            'label': text,
                            'writer_id': writer_id,
                            'level': level,
                            'split': split
                        })
                except json.JSONDecodeError:
                    print(f"Error reading {json_path}")
                    
    return pd.DataFrame(records)

base_dataset_path = next((os.path.join('/kaggle/input', d) for d in os.listdir('/kaggle/input') if os.path.isdir(os.path.join('/kaggle/input', d))), 'VietHandOCR_Datasets') if os.path.exists('/kaggle/input') else 'VietHandOCR_Datasets'
metadata = build_metadata(base_dataset_path)
metadata = reduce_mem_usage(metadata)

train_val_data = metadata[metadata['split'] == 'train_data']
test_data = metadata[metadata['split'] == 'test_data']

train_df, val_df = writer_independent_split(train_val_data)

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_data)}")
train_df[['image_path', 'label']].to_csv('train.txt', sep='\t', index=False, header=False)
val_df[['image_path', 'label']].to_csv('val.txt', sep='\t', index=False, header=False)
test_data[['image_path', 'label']].to_csv('test.txt', sep='\t', index=False, header=False)
del metadata, train_val_data, train_df, val_df, test_data
gc.collect()


Memory decreased from 4.53MB to 1.93MB
Train size: 103724, Val size: 12024, Test size: 3113


0